# Balanceo global y regeneración controlada de casos PERT-BAL

Este notebook reduce el sesgo de la muestra y genera casos PERT-BAL directamente en PostgreSQL. Usa perfiles `USER-*` diversos, selecciona carreras de la misma área con mayor similitud Jaccard y ajusta dropout/degradación por banda sin asignar targets artificiales.

La persistencia es idempotente: cada candidato usa un UUID determinista y se inserta dentro de una transacción. No se crean archivos CSV ni se implementa `usuario_origen`; la procedencia se conserva temporalmente en el prefijo y en la columna `uuid_origen` del resumen en memoria.

## 1. Configuración

La ejecución escribe únicamente en la base de datos configurada aquí. El límite por estrato controla la muestra de entrenamiento; las cuotas PERT controlan la regeneración de casos y se aplican por área y banda.

In [1]:
from pathlib import Path
import hashlib
import json
import math
import random
from typing import Optional

import numpy as np
import pandas as pd
import sqlalchemy as sa
from faker import Faker
from IPython.display import display

DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = 'password'
DB_CONECTION = 'athena'

connection_url = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url, connect_args={'connect_timeout': 10})
fake = Faker('es_MX')

SEED = 20260928
MAX_PER_STRATUM = 100
PERSIST_CANDIDATES = True
CANDIDATES_PER_BAND = 10
CAREERS_PER_PROFILE = 8
random.seed(SEED)
np.random.seed(SEED)

## 2. Extracción de resultados y recomputación del target

La consulta usa únicamente `SELECT`. Consolida habilidades repetidas con `MAX`, limita el cruce a la misma área y conserva el target persistido para auditarlo contra `target_recalculado`.

In [2]:
QUERY_DATASET = '''
WITH perfil AS (
    SELECT
        u.id_usuario,
        u.uuid_usuario,
        u.id_area,
        uh.id_habilidad,
        MAX(COALESCE(uh.cumplimiento_criterio, 0.0))::DOUBLE PRECISION AS cumplimiento
    FROM usuario u
    JOIN usuario_habilidad uh ON uh.id_usuario = u.id_usuario
    GROUP BY u.id_usuario, u.uuid_usuario, u.id_area, uh.id_habilidad
), requerimientos AS (
    SELECT
        c.id_area,
        c.id_carrera,
        c.nombre_carrera,
        ch.id_habilidad,
        COUNT(cr.id_criterio)::INTEGER AS complejidad_habilidad
    FROM carrera c
    JOIN carrera_habilidad ch ON ch.id_carrera = c.id_carrera
    LEFT JOIN criterio cr ON cr.id_habilidad = ch.id_habilidad
    GROUP BY c.id_area, c.id_carrera, c.nombre_carrera, ch.id_habilidad
), calculado AS (
    SELECT
        p.id_usuario, p.uuid_usuario, p.id_area,
        r.id_carrera, r.nombre_carrera,
        COUNT(*)::INTEGER AS habilidades_requeridas,
        COUNT(pm.id_habilidad)::INTEGER AS habilidades_comunes,
        COALESCE(SUM(pm.cumplimiento), 0.0)::DOUBLE PRECISION AS suma_cumplimiento,
        COALESCE(SUM(r.complejidad_habilidad), 0)::INTEGER AS complejidad_requerida,
        COALESCE(SUM(CASE WHEN pm.id_habilidad IS NOT NULL THEN r.complejidad_habilidad ELSE 0 END), 0)::INTEGER AS complejidad_cumplida,
        COALESCE(SUM(pm.cumplimiento), 0.0) / NULLIF(COUNT(*), 0)::DOUBLE PRECISION AS target_recalculado
    FROM perfil p
    JOIN requerimientos r ON r.id_area = p.id_area
    LEFT JOIN perfil pm
      ON pm.id_usuario = p.id_usuario
     AND pm.id_habilidad = r.id_habilidad
    GROUP BY p.id_usuario, p.uuid_usuario, p.id_area, r.id_carrera, r.nombre_carrera
)
SELECT
    c.*,
    r.porcentaje_coincidencia::DOUBLE PRECISION AS target_persistido,
    r.top,
    CASE
      WHEN c.uuid_usuario LIKE 'USER-%' THEN 'USER'
      WHEN c.uuid_usuario LIKE 'CROSS-%' THEN 'CROSS'
      WHEN c.uuid_usuario LIKE 'PERT-BAL-%' THEN 'PERT-BAL'
      WHEN c.uuid_usuario LIKE 'PERT-%' THEN 'PERT'
      WHEN c.uuid_usuario LIKE 'COMBO-%' THEN 'COMBO'
      ELSE 'OTRA'
    END AS estrategia
FROM calculado c
JOIN resultado r
  ON r.uuid_usuario = c.uuid_usuario
 AND r.id_carrera = c.id_carrera
WHERE r.top = 1
ORDER BY c.id_area, c.id_usuario, c.id_carrera;
'''

with engine.connect() as connection:
    df = pd.read_sql_query(sa.text(QUERY_DATASET), connection)

if df.empty:
    raise ValueError('La consulta no devolvio resultados.')

df['target_recalculado_2dp'] = (
    np.floor(df['target_recalculado'] * 100 + 0.500000001) / 100
)
df['diferencia_target'] = df['target_persistido'] - df['target_recalculado_2dp']
df['decil'] = np.minimum(np.floor(df['target_persistido'] * 10) / 10, 0.9)
df.loc[df['target_persistido'] >= 1.0, 'decil'] = 1.0
df['decil'] = df['decil'].round(1)
df['grupo_origen'] = df['uuid_usuario']

print(f'Resultados extraidos: {len(df):,}')
display(df.head())

Resultados extraidos: 7,997


,id_usuario,uuid_usuario,id_area,id_carrera,nombre_carrera,habilidades_requeridas,habilidades_comunes,suma_cumplimiento,complejidad_requerida,complejidad_cumplida,target_recalculado,target_persistido,top,estrategia,target_recalculado_2dp,diferencia_target,decil,grupo_origen
0,1,USER-6cd5bff9-11ca-4897-b6db-5ea97a9777cc,1,1,Ingeniería Aeronáutica,49,49,49.0,147,147,1.0,1.0,1,USER,1.0,0.0,1.0,USER-6cd5bff9-11ca-4897-b6db-5ea97a9777cc
1,2,USER-676e8140-5883-4730-a238-a4aebbb617cd,1,2,Ingeniería Ambiental,36,36,36.0,108,108,1.0,1.0,1,USER,1.0,0.0,1.0,USER-676e8140-5883-4730-a238-a4aebbb617cd
2,3,USER-e2457dd4-2455-446f-b71c-2bbb93a3e643,1,3,Ingeniería Biomédica,36,36,36.0,108,108,1.0,1.0,1,USER,1.0,0.0,1.0,USER-e2457dd4-2455-446f-b71c-2bbb93a3e643
3,4,USER-62482fe2-fc6a-42a9-9e0f-a066164798f4,1,4,Ingeniería Biónica,36,36,36.0,102,102,1.0,1.0,1,USER,1.0,0.0,1.0,USER-62482fe2-fc6a-42a9-9e0f-a066164798f4
4,5,USER-a52ccc02-c05b-4582-b9c0-e3ba4dbfeb92,1,5,Ingeniería Bioquímica,36,36,36.0,102,102,1.0,1.0,1,USER,1.0,0.0,1.0,USER-a52ccc02-c05b-4582-b9c0-e3ba4dbfeb92


## 5. Variación de perfiles y carreras similares, sin persistencia

Se cargan perfiles `USER-*`, habilidades de carreras del mismo área y se calcula Jaccard. Para cada banda se prueban dropout y degradación compatibles con el tamaño del perfil; si no se alcanza la banda, el candidato se descarta. El target siempre se recalcula con las habilidades presentes.

In [3]:
PROFILE_QUERY = '''
SELECT u.uuid_usuario, u.id_area, uh.id_habilidad,
       MAX(COALESCE(uh.cumplimiento_criterio, 0.0))::DOUBLE PRECISION AS cumplimiento
FROM usuario u
JOIN usuario_habilidad uh ON uh.id_usuario = u.id_usuario
WHERE u.uuid_usuario LIKE 'USER-%'
GROUP BY u.uuid_usuario, u.id_area, uh.id_habilidad
ORDER BY u.uuid_usuario, uh.id_habilidad;
'''
CAREER_QUERY = '''
SELECT c.id_area, c.id_carrera, c.nombre_carrera, ch.id_habilidad
FROM carrera c
JOIN carrera_habilidad ch ON ch.id_carrera = c.id_carrera
ORDER BY c.id_area, c.id_carrera, ch.id_habilidad;
'''
with engine.connect() as connection:
    profile_rows = connection.execute(sa.text(PROFILE_QUERY)).mappings().all()
    career_rows = connection.execute(sa.text(CAREER_QUERY)).mappings().all()

profiles = {}
for row in profile_rows:
    profile = profiles.setdefault(row['uuid_usuario'], {
        'uuid_usuario': row['uuid_usuario'],
        'id_area': row['id_area'],
        'habilidades': {}
    })
    profile['habilidades'][row['id_habilidad']] = float(row['cumplimiento'])

careers = {}
for row in career_rows:
    career = careers.setdefault(row['id_carrera'], {
        'id_area': row['id_area'],
        'id_carrera': row['id_carrera'],
        'nombre_carrera': row['nombre_carrera'],
        'habilidades': set()
    })
    career['habilidades'].add(row['id_habilidad'])

def jaccard(left: set[int], right: set[int]) -> float:
    union = left | right
    return len(left & right) / len(union) if union else 0.0

def candidate_bands() -> list[dict]:
    return [
        {'nombre_banda': 'media', 'lower': 0.50, 'upper': 0.60, 'dropout': (1, 2), 'degradacion': (0.40, 0.70)},
        {'nombre_banda': 'media_alta', 'lower': 0.60, 'upper': 0.70, 'dropout': (0, 1), 'degradacion': (0.55, 0.80)},
        {'nombre_banda': 'alta', 'lower': 0.70, 'upper': 0.80, 'dropout': (0, 1), 'degradacion': (0.70, 0.90)},
        {'nombre_banda': 'muy_alta', 'lower': 0.80, 'upper': 0.90, 'dropout': (0, 0), 'degradacion': (0.75, 0.95)},
    ]

def deterministic_uuid(source_uuid: str, career_id: int, band: str, attempt: int) -> str:
    payload = f'{source_uuid}|{career_id}|{band}|{attempt}'
    digest = hashlib.sha256(payload.encode('utf-8')).hexdigest()[:32]
    return f'PERT-BAL-{digest}'

def build_candidates() -> pd.DataFrame:
    rng = random.Random(SEED)
    output = []
    profiles_by_area = {}
    careers_by_area = {}
    bands = candidate_bands()
    for profile in profiles.values():
        profiles_by_area.setdefault(profile['id_area'], []).append(profile)
    for career in careers.values():
        careers_by_area.setdefault(career['id_area'], []).append(career)

    for area, area_profiles in profiles_by_area.items():
        band_counts = {band['nombre_banda']: 0 for band in bands}
        area_careers = careers_by_area.get(area, [])
        for profile in area_profiles:
            if all(count >= CANDIDATES_PER_BAND for count in band_counts.values()):
                break
            profile_skills = set(profile['habilidades'])
            ranked = sorted(
                area_careers,
                key=lambda career: jaccard(profile_skills, career['habilidades']),
                reverse=True
            )[:CAREERS_PER_PROFILE]
            for career in ranked:
                if all(count >= CANDIDATES_PER_BAND for count in band_counts.values()):
                    break
                required = career['habilidades']
                if not required or not required.intersection(profile_skills):
                    continue
                similarity = jaccard(profile_skills, required)
                for band in bands:
                    band_name = band['nombre_banda']
                    if band_counts[band_name] >= CANDIDATES_PER_BAND:
                        continue
                    for attempt in range(1, 101):
                        varied = dict(profile['habilidades'])
                        low_drop, high_drop = band['dropout']
                        dropout = min(rng.randint(low_drop, high_drop), max(0, len(varied) - 1))
                        if dropout:
                            for skill_id in rng.sample(list(varied), dropout):
                                del varied[skill_id]
                        degradable = list(required.intersection(varied))
                        if degradable:
                            skill_id = rng.choice(degradable)
                            low_value, high_value = band['degradacion']
                            varied[skill_id] = round(rng.uniform(low_value, high_value), 2)
                        common = required.intersection(varied)
                        target = round(sum(varied[s] for s in common) / len(required), 4)
                        if band['lower'] <= target < band['upper']:
                            digest = hashlib.sha256(profile['uuid_usuario'].encode()).hexdigest()[:12]
                            output.append({
                                'uuid_usuario': deterministic_uuid(profile['uuid_usuario'], career['id_carrera'], band_name, attempt),
                                'uuid_origen': profile['uuid_usuario'],
                                'nombre': fake.name(),
                                'correo': 'pert_' + digest + '_' + str(band_counts[band_name]) + '@example.invalid',
                                'id_area': area,
                                'id_carrera': career['id_carrera'],
                                'carrera': career['nombre_carrera'],
                                'similitud_jaccard': round(similarity, 4),
                                'banda': band_name,
                                'habilidades': varied,
                                'habilidades_presentes': len(varied),
                                'habilidades_comunes': len(common),
                                'target_recalculado': target,
                                'estrategia': 'PERT-BAL'
                            })
                            band_counts[band_name] += 1
                            break
    return pd.DataFrame(output)

def persist_candidates(frame: pd.DataFrame) -> None:
    if not PERSIST_CANDIDATES or frame.empty:
        return
    with engine.begin() as connection:
        for candidate in frame.to_dict('records'):
            connection.execute(sa.text('''
                INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
                VALUES (:nombre, :correo, :uuid_usuario, :id_area)
                ON CONFLICT (uuid_usuario) DO NOTHING;
            '''), candidate)
            for skill_id, completion in candidate['habilidades'].items():
                connection.execute(sa.text('''
                    INSERT INTO usuario_habilidad
                        (id_usuario, id_habilidad, cumplimiento_criterio)
                    SELECT u.id_usuario, :id_habilidad, :cumplimiento
                    FROM usuario u
                    WHERE u.uuid_usuario = :uuid_usuario
                      AND NOT EXISTS (
                          SELECT 1 FROM usuario_habilidad uh
                          WHERE uh.id_usuario = u.id_usuario
                            AND uh.id_habilidad = :id_habilidad
                      );
                '''), {
                    'uuid_usuario': candidate['uuid_usuario'],
                    'id_habilidad': skill_id,
                    'cumplimiento': completion
                })
            connection.execute(sa.text('''
                INSERT INTO resultado
                    (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
                VALUES (:uuid_usuario, :id_carrera, :target_recalculado, 1)
                ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                    porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                    top = EXCLUDED.top;
            '''), candidate)

fake.seed_instance(SEED)
candidates = build_candidates()
persist_candidates(candidates)
print(f'Candidatos PERT-BAL generados: {len(candidates):,}')
if not candidates.empty:
    display(candidates.drop(columns=['habilidades']).head())

Candidatos PERT-BAL generados: 60


,uuid_usuario,uuid_origen,nombre,correo,id_area,id_carrera,carrera,similitud_jaccard,banda,habilidades_presentes,habilidades_comunes,target_recalculado,estrategia
0,PERT-BAL-fa5610f4f48300f3dbd5730720fbb952,USER-04bd3876-82d6-4c37-a52e-cf09a1621253,Eugenia Aparicio,pert_71a61399551e_0@example.invalid,3,57,Contador Público,1.0,media,3,3,0.528,PERT-BAL
1,PERT-BAL-8e2f7cbfc89127e7b7e2931990dbe680,USER-04bd3876-82d6-4c37-a52e-cf09a1621253,Bernardo Rael,pert_71a61399551e_0@example.invalid,3,57,Contador Público,1.0,alta,4,4,0.766,PERT-BAL
2,PERT-BAL-418c89e783f1c1213af6bdf9799e3068,USER-2a878df4-1d96-4fc9-aab6-3cf9c0c81190,Rosalia Estévez Meza,pert_3354aef2e098_1@example.invalid,3,66,Licenciatura en Turismo,1.0,media,3,3,0.536,PERT-BAL
3,PERT-BAL-24973dbe537cfbd3201431733d7b36de,USER-2a878df4-1d96-4fc9-aab6-3cf9c0c81190,Alma Ríos,pert_3354aef2e098_1@example.invalid,3,66,Licenciatura en Turismo,1.0,alta,4,4,0.766,PERT-BAL
4,PERT-BAL-c717165fc108285781e9bf99e2e1782d,USER-3942a43c-eaff-4b0a-9d93-0953741a7dd7,Gregorio Padilla Velásquez,pert_9c1821cb4942_2@example.invalid,3,56,Licenciatura en Administración Industrial,1.0,media,3,3,0.534,PERT-BAL


## 6. Validaciones y exportación

Los CSV exportados son artefactos derivados para entrenamiento y auditoría. La validación informa escasez de estratos y verifica targets, áreas y reproducibilidad.

In [5]:
def stratified_sample(frame: pd.DataFrame, max_per_stratum: int, seed: int) -> pd.DataFrame:
    groups = []
    for _, group in frame.groupby(['id_area', 'decil'], sort=True):
        groups.append(group.sample(n=min(len(group), max_per_stratum), random_state=seed))
    return pd.concat(groups, ignore_index=True) if groups else frame.iloc[0:0].copy()

def validate_balanced_sample(frame: pd.DataFrame) -> dict:
    if not frame['target_persistido'].between(0, 1).all():
        raise AssertionError('Hay targets fuera del intervalo [0, 1].')
    if (frame['diferencia_target'].abs() > 0.011).any():
        raise AssertionError('El target persistido no coincide con el recalculo a 2 decimales.')
    return {
        'filas_balanceadas': int(len(frame)),
        'grupos_origen': int(frame['grupo_origen'].nunique()),
        'pct_decil_cero': float((frame['decil'] == 0.0).mean() * 100),
        'pct_area_mayoritaria': float(frame['id_area'].value_counts(normalize=True).max() * 100),
        'targets_fuera_de_rango': int((~frame['target_persistido'].between(0, 1)).sum()),
    }

balanced = stratified_sample(df, MAX_PER_STRATUM, SEED)
metrics = validate_balanced_sample(balanced)
display(pd.DataFrame([metrics]))

if not candidates.empty:
    if not candidates['target_recalculado'].between(0, 1).all():
        raise AssertionError('Hay candidatos PERT-BAL fuera del intervalo [0, 1].')
    band_summary = (
        candidates.groupby(['id_area', 'banda'], as_index=False)
        .agg(casos=('uuid_usuario', 'size'), target_medio=('target_recalculado', 'mean'))
    )
    display(band_summary)
    expected_bands = {'media', 'media_alta', 'alta', 'muy_alta'}
    missing_bands = expected_bands.difference(candidates['banda'])
    if missing_bands:
        print('Bandas no alcanzables con los perfiles actuales:', sorted(missing_bands))

with engine.connect() as connection:
    persisted_count = connection.execute(sa.text('''
        SELECT COUNT(*)
        FROM usuario
        WHERE uuid_usuario LIKE 'PERT-BAL-%';
    ''')).scalar_one()
print(f'Usuarios PERT-BAL persistidos: {persisted_count:,}')
print('No se generaron archivos CSV.')

,filas_balanceadas,grupos_origen,pct_decil_cero,pct_area_mayoritaria,targets_fuera_de_rango
0,1574,1574,19.05972,43.138501,0


,id_area,banda,casos,target_medio
0,1,alta,10,0.7588
1,1,media,10,0.5160
2,2,alta,10,0.7574
3,2,media,10,0.5268
4,3,alta,10,0.7584
5,3,media,10,0.5198


Bandas no alcanzables con los perfiles actuales: ['media_alta', 'muy_alta']
Usuarios PERT-BAL persistidos: 362
No se generaron archivos CSV.


## Decisiones de uso

- El undersampling no modifica `usuario`, `usuario_habilidad` ni `resultado`.
- Los candidatos PERT no se insertan y no deben mezclarse con el dataset persistido hasta revisar sus métricas.
- Si el decil 0 o un área siguen superando el criterio, no se inventan filas: se reporta la escasez real y se ajustan cuotas o la generación en una iteración posterior.
- Una tabla persistente de procedencia sigue siendo la solución definitiva para reemplazar el manifiesto.